# 00.2 · The AI family tree

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/00-big-picture/02-ai-family-tree/lesson.ipynb)

Read the [README](README.md) first. This notebook runs one tiny example from each box of the family tree, on the same ten orders, so you can see that they differ in *what they learn from*, not in how the code looks.

Do not try to understand the neural network or LLM cells. Look at their shape.

## The data: ten orders, one question

Will the customer return within 90 days?

In [1]:
import pandas as pd

# ten ShopWave orders: will the customer return within 90 days?
orders = pd.DataFrame({
    "total":         [45, 620, 310, 880, 95, 540, 430, 70, 150, 260],
    "items":         [2, 9, 4, 12, 1, 6, 7, 3, 2, 5],
    "account_days":  [400, 3, 120, 1, 2, 800, 5, 30, 60, 200],
    "returns":       [1, 1, 1, 1, 0, 1, 1, 0, 0, 1],   # 1 = came back within 90 days
})
X = orders[["total", "items", "account_days"]]
y = orders["returns"]
orders

,total,items,account_days,returns
0,45,2,400,1
1,620,9,3,1
2,310,4,120,1
3,880,12,1,1
4,95,1,2,0
5,540,6,800,1
6,430,7,5,1
7,70,3,30,0
8,150,2,60,0
9,260,5,200,1


## Box 1: AI without learning, a rule

Someone writes the rule. Nothing is learned from the table.

In [2]:
def rule(row):
    return int(row["total"] > 100)

pred_rule = orders.apply(rule, axis=1)
print(f"rule accuracy: {(pred_rule == y).mean():.0%}")

rule accuracy: 80%


## Box 2: machine learning, a decision tree

The model learns thresholds from `X` and `y`. Same table, no human-written rule.

In [3]:
from sklearn.tree import DecisionTreeClassifier, export_text

tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X, y)
print(f"tree accuracy: {(tree.predict(X) == y).mean():.0%}")
print(export_text(tree, feature_names=list(X.columns)))

tree accuracy: 100%
|--- items <= 3.50
|   |--- account_days <= 230.00
|   |   |--- class: 0
|   |--- account_days >  230.00
|   |   |--- class: 1
|--- items >  3.50
|   |--- class: 1



## Box 3: deep learning, a (very small) neural network

Also `fit(X, y)`. Inside, instead of if/else questions, it is a stack of layers of weighted sums. On ten rows of numbers this is silly, and you will see why in Module 08. The point here: **it is still a model that takes `X` and `y`.**

(The inputs are scaled first; neural networks need that. Module 04 explains.)

In [4]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings("ignore")

X_scaled = StandardScaler().fit_transform(X)
net = MLPClassifier(hidden_layer_sizes=(8, 8), max_iter=2000, random_state=0).fit(X_scaled, y)
print(f"neural net accuracy: {(net.predict(X_scaled) == y).mean():.0%}")
print("layers of weights:", [w.shape for w in net.coefs_])

neural net accuracy: 100%
layers of weights: [(3, 8), (8, 8), (8, 1)]


Three layers of weights (3→8, 8→8, 8→1): over a hundred numbers (weights and biases) learned from 10 rows. A decision tree learned two thresholds. Both fit the training data. Which would you trust on an eleventh order?

## Box 4: generative AI, a language model

Different shape: **text in, text out**. The model was trained by someone else on a huge amount of text; we only ask. If you have no API key and no local model, the cell explains that and moves on.

In [5]:
import os

prompt = "In two sentences, explain to a shop manager what machine learning is."

def ask_llm(prompt):
    """Try the Anthropic API, then a local Ollama model, then give up gracefully."""
    key = os.environ.get("ANTHROPIC_API_KEY")
    if key:
        try:
            import anthropic
            client = anthropic.Anthropic(api_key=key)
            msg = client.messages.create(model="claude-sonnet-5-5", max_tokens=200,
                                         messages=[{"role": "user", "content": prompt}])
            return "Anthropic API: " + msg.content[0].text
        except Exception as e:
            print("Anthropic call failed:", e)
    try:
        import requests
        r = requests.post("http://localhost:11434/api/generate",
                          json={"model": "llama3.2", "prompt": prompt, "stream": False}, timeout=60)
        return "Local Ollama: " + r.json()["response"]
    except Exception:
        pass
    return ("No LLM available in this environment (no ANTHROPIC_API_KEY and no local Ollama). "
            "That is fine for now: Module 09 sets this up properly. "
            "An LLM would answer the prompt with two fluent sentences it generated word by word.")

print(ask_llm(prompt))

No LLM available in this environment (no ANTHROPIC_API_KEY and no local Ollama). That is fine for now: Module 09 sets this up properly. An LLM would answer the prompt with two fluent sentences it generated word by word.


## On ShopWave's real columns

The real data has: invoice, product code, description, quantity, date, price, customer ID, country.

| Problem | Box |
|---|---|
| Predict next quarter's spend from past orders | ML |
| Group customers by buying pattern | ML (unsupervised) |
| Flag unusual orders | ML (anomaly detection) |
| Forecast monthly sales | ML (time series) |
| Read the free-text product *descriptions* and group similar products | borderline: classical text ML works; deep learning better |
| Tag product *photos* (when ShopWave adds them) | deep learning |
| Answer customer support chats | generative AI |

Almost everything about a table of numbers is classical ML. That is why Modules 01–07 come first.

## Quiz answers

1. The **VAT formula**. It is a rule, not learned from data.
2. Yes, every DL model is an ML model (inner box). No, most ML models (trees, k-means, linear regression) are not deep learning.
3. *"What does it learn from, and what does it output?"*
4. "A two-year table of sales is small, structured data. Classical ML (gradient-boosted trees, or even a seasonal baseline) will likely do as well, train in seconds, and be explainable. Let us start there and only go deeper if it fails."
5. The number of **layers** of neurons stacked between input and output.

**Next:** [00.3 The three kinds of learning](../03-three-kinds-of-learning/)